# Live Camera Distraction Detection

Shows a genuinely **continuous live video feed** from the board's USB webcam, with the current frame rate and the latest detected class overlaid directly on the frame. Capture/display and model inference run as two independent loops (a background thread does inference) so the video itself keeps updating smoothly instead of freezing for ~9 seconds between each detection.

**Why two loops:** inference on this board takes ~9s/frame (XNNPACK is disabled on this ARM32 build for safety). A single capture-then-infer-then-display loop can only show a new frame once every ~9 seconds - a slideshow, not a live feed. Here, a background thread keeps running inference on whatever the latest frame happens to be, while the main loop keeps grabbing frames and updating the on-screen image many times a second regardless of whether a new detection has landed yet. The overlaid class label only changes about once every 9 seconds (that's still how often a *new* detection exists), but the picture itself stays live, and an on-screen `fps` counter shows exactly how live.

**Camera:** auto-detected by its own UVC descriptor name (`HDF Webcam USB`), not a hardcoded `/dev/videoN` path - device numbering has drifted before on this board (video0, video2, and elsewhere across sessions), so a fixed path silently breaks after a reboot or a USB change.

**Alert logic + images:** uses `AlertHysteresis` imported directly from `alert_loop_infer.py` (already on this board at `/home/xilinx/alert_loop_infer.py`) - the same 8-consecutive-confident-tick logic verified against `fpga/rtl/distraction_alert_controller.v`'s testbench. A real alert POSTs to the VM backend with the actual frame that triggered it (JPEG, base64), so the Android app can show what the driver was doing, not just a text label.

**How to stop:** use Jupyter's Interrupt Kernel (■) button at any time, or it stops on its own after `RUN_SECONDS`.


In [ ]:
import time
import threading
from collections import deque
import numpy as np
import cv2
from tflite_runtime import interpreter as tflite
import ipywidgets as widgets
from IPython.display import display

from alert_loop_infer import AlertHysteresis, LABEL_NAMES, CONFIDENCE_THRESHOLD, \
    WEBCAM_NAME_MATCH, find_camera_device, post_alert, sound_buzzer_alert

MODEL_PATH = "/home/xilinx/mobilenetv2_crossview_finetuned_int8.tflite"
RUN_SECONDS = 300  # how long the live feed runs for (5 min default) - stop earlier with Interrupt Kernel
DISPLAY_INTERVAL_S = 0.03  # target loop pace; actual achieved rate is measured and shown on-screen
JPEG_QUALITY = 70  # lower = faster encode, smaller payload to the browser, a bit blockier picture
DISPLAY_WIDTH, DISPLAY_HEIGHT = 320, 240  # capture at this size directly (not resized after) - the
                                           # single biggest lever against lag, since every later step
                                           # (copy, putText, JPEG encode, browser transfer) scales with it

CAMERA_DEVICE = find_camera_device(WEBCAM_NAME_MATCH)
print(f"Using camera: {CAMERA_DEVICE}")

interp = tflite.Interpreter(model_path=MODEL_PATH)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print("Model loaded:", MODEL_PATH)


In [ ]:
cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
if not cap.isOpened():
    raise RuntimeError(f"Could not open {CAMERA_DEVICE}")

# Ask the camera to capture at a smaller size directly, rather than capturing
# full-size and downscaling every frame in software - most UVC webcams support
# this natively. Not all devices honour it, so read back what we actually got.
cap.set(cv2.CAP_PROP_FRAME_WIDTH, DISPLAY_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, DISPLAY_HEIGHT)
actual_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
actual_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Requested {DISPLAY_WIDTH}x{DISPLAY_HEIGHT}, camera is actually giving {actual_w}x{actual_h}")

hysteresis = AlertHysteresis()
state_lock = threading.Lock()
latest_frame = None  # most recent captured frame (BGR), shared with the inference thread
latest_result = {"class_id": 0, "confidence": 0.0, "infer_time_s": 0.0, "tick": 0,
                  "distracted_run": 0, "safe_run": 0, "alert": False}
stop_event = threading.Event()


def inference_worker():
    '''Runs continuously in the background: grabs whatever the latest captured frame is,
    classifies it, updates the shared result dict, and fires alerts (with that same
    frame attached) - independently of how fast the display loop below is refreshing
    the picture on screen.'''
    while not stop_event.is_set():
        with state_lock:
            frame = None if latest_frame is None else latest_frame.copy()
        if frame is None:
            time.sleep(0.05)
            continue

        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LINEAR)
        x = resized.astype(inp['dtype'])[None, ...]

        t0 = time.time()
        interp.set_tensor(inp['index'], x)
        interp.invoke()
        y = interp.get_tensor(out['index'])[0]
        elapsed = time.time() - t0

        class_id = int(np.argmax(y))
        confidence = float(y[class_id])
        fired = hysteresis.tick(class_id, confidence)

        with state_lock:
            latest_result.update({
                "class_id": class_id, "confidence": confidence, "infer_time_s": elapsed,
                "tick": latest_result["tick"] + 1,
                "distracted_run": hysteresis.distracted_run,
                "safe_run": hysteresis.safe_run, "alert": hysteresis.alert,
            })

        if fired:
            print(f"*** ALERT FIRED: {LABEL_NAMES[class_id]} ({confidence:.0%}) - posting to backend with image ***")
            sound_buzzer_alert()
            post_alert(LABEL_NAMES[class_id], confidence, frame_bgr=frame)


worker = threading.Thread(target=inference_worker, daemon=True)
worker.start()
print("Inference worker started.")


In [ ]:
image_widget = widgets.Image(format='jpeg', width=DISPLAY_WIDTH, height=DISPLAY_HEIGHT)
display(image_widget)

encode_params = [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY]
frame_times = deque(maxlen=30)  # rolling window for a smoothed, honest fps reading

start_time = time.time()
frames_shown = 0

try:
    while time.time() - start_time < RUN_SECONDS:
        loop_t0 = time.time()

        ret, frame = cap.read()
        if not ret:
            time.sleep(0.05)
            continue

        with state_lock:
            latest_frame = frame
            result = dict(latest_result)

        frame_times.append(loop_t0)
        if len(frame_times) >= 2:
            fps = (len(frame_times) - 1) / (frame_times[-1] - frame_times[0])
        else:
            fps = 0.0

        label = LABEL_NAMES[result["class_id"]]
        conf = result["confidence"]
        alert_active = result["alert"]
        color = (0, 0, 255) if alert_active else (0, 200, 0)  # BGR: red when alerting, green otherwise

        display_frame = frame.copy()
        cv2.putText(display_frame, f"{label} ({conf:.0%})", (8, 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        status_text = "ALERT" if alert_active else f"d={result['distracted_run']} s={result['safe_run']}"
        cv2.putText(display_frame, status_text, (8, 45),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        cv2.putText(display_frame, f"{fps:.1f} fps", (8, DISPLAY_HEIGHT - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)

        ok, jpeg = cv2.imencode('.jpg', display_frame, encode_params)
        if ok:
            image_widget.value = jpeg.tobytes()
            frames_shown += 1

        elapsed_this_loop = time.time() - loop_t0
        remaining = DISPLAY_INTERVAL_S - elapsed_this_loop
        if remaining > 0:
            time.sleep(remaining)

except KeyboardInterrupt:
    print("Stopped by user.")

finally:
    stop_event.set()
    worker.join(timeout=2)
    cap.release()
    elapsed_total = time.time() - start_time
    print(f"\nStopped after {elapsed_total:.0f}s: displayed {frames_shown} frames "
          f"({frames_shown / elapsed_total:.1f} fps average), ran {latest_result['tick']} detection tick(s).")
